# Ablation & ensembling results

Analysis of the prompt-length ablation (k ∈ {5, 10, 20}), the mBERT vs XLM-R comparison,
length-normalised scoring and soft-prompt ensembling, for both inflection variants.

**Where the inputs come from** (all paths relative to the project root, as written by the Slurm jobs):

| What | Path | Written by |
|---|---|---|
| Per-prompt evaluation (token accuracy vs zero-shot) | `results/ablation_<variant>/<model>/<lang>/eval_P<k>_seed<s>.json` | `ablation_train.slurm` → `prompt_tuning_eval.py` |
| Candidate-ranking EM for every member, α and ensemble | `results/ablation_<variant>/<model>/<lang>/ensemble.json` | `ablation_ensemble.slurm` → `prompt_ensemble.py` |
| Training curves | `checkpoints/ablation_<variant>/<model>/<lang>/P<k>_seed<s>/history_<lang>_P<k>.json` | `prompt_tuning_train.py` |
| Discrete-template baseline (optional) | `results/baseline_probe/summary.tsv` | `baseline_probe.slurm` |

`<variant>` is `keep` (all facts, Russian lowercased) or `drop` (facts with malformed ru/el inflections removed).

**Running it.** On Snellius, open it from the project folder (`pip install jupyter` in the env if needed).
To run on your laptop instead, copy the result folders first, from the project root:

```bash
scp -r scur0540@snellius.surf.nl:~/Multilingual_NLP_Project/results/ablation_keep  results/
scp -r scur0540@snellius.surf.nl:~/Multilingual_NLP_Project/results/ablation_drop  results/
scp -r scur0540@snellius.surf.nl:~/Multilingual_NLP_Project/results/baseline_probe results/
# training curves only (small JSON files, skips the .pt checkpoints):
rsync -av --include='*/' --include='history_*.json' --exclude='*' \
    scur0540@snellius.surf.nl:~/Multilingual_NLP_Project/checkpoints/ ./checkpoints/
```

Figures are saved to `results/analysis_figures/`.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display, Markdown

# Project root = the folder that contains pipeline_v1/ (works from any subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "pipeline_v1").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

VARIANTS = {
    "keep": {"results": ROOT / "results" / "ablation_keep",
             "checkpoints": ROOT / "checkpoints" / "ablation_keep"},
    "drop": {"results": ROOT / "results" / "ablation_drop",
             "checkpoints": ROOT / "checkpoints" / "ablation_drop"},
    # Follow-up runs (one model each); missing folders are skipped.
    "xlmr_lr0.05": {"results": ROOT / "results" / "ablation_keep_xlmr_lr005",
                    "checkpoints": ROOT / "checkpoints" / "ablation_keep_xlmr_lr005"},
    "mbert_ep20": {"results": ROOT / "results" / "ablation_keep_mbert_ep20",
                   "checkpoints": ROOT / "checkpoints" / "ablation_keep_mbert_ep20"},
}
BASELINE = ROOT / "results" / "baseline_probe" / "summary.tsv"
FIG_DIR = ROOT / "results" / "analysis_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Variant used for the single-variant sections (length normalisation, ensembles, curves).
FOCUS = "keep"

print(f"Project root: {ROOT}")
for name, paths in VARIANTS.items():
    for kind, path in paths.items():
        print(f"  {name:<5} {kind:<12} {'found  ' if path.is_dir() else 'MISSING'} {path}")
print(f"  baseline           {'found  ' if BASELINE.exists() else 'MISSING'} {BASELINE}")

In [ ]:
# --- Colours (validated categorical palette; identity follows the entity) ---
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
               "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
LANG_ORDER = ["en", "nl", "ru", "el", "ko"]
MODEL_ORDER = ["mbert_base", "xlmr_base"]
MODEL_LABELS = {"mbert_base": "mBERT (WordPiece)", "xlmr_base": "XLM-R (SentencePiece)"}
MODEL_COLORS = {"mbert_base": "#4a3aa7", "xlmr_base": "#008300"}
SYSTEM_ORDER = ["zero_shot", "best_single", "uniform", "learned"]
SYSTEM_LABELS = {"zero_shot": "Zero-shot", "best_single": "Best single prompt",
                 "uniform": "Uniform ensemble", "learned": "Learned weights"}
SYSTEM_COLORS = dict(zip(SYSTEM_ORDER, CATEGORICAL[:4]))
# k is ordinal, so it gets one sequential hue (light = short prompt, dark = long).
K_RAMP = ["#86b6ef", "#2a78d6", "#104281", "#0d366b"]
BLUES = LinearSegmentedColormap.from_list("blues", ["#f0f6fd", "#9ec5f4", "#2a78d6", "#0d366b"])

INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "font.size": 10,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "axes.grid": True, "axes.grid.axis": "y", "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "lines.markersize": 6, "legend.frameon": False,
    "legend.labelcolor": INK2, "figure.dpi": 110,
})


def ordered(values, order):
    """Known values in the preferred order, then any others alphabetically."""
    values = list(dict.fromkeys(values))
    return [v for v in order if v in values] + sorted(v for v in values if v not in order)


def lang_colors(langs):
    return {lang: CATEGORICAL[i % len(CATEGORICAL)] for i, lang in enumerate(ordered(langs, LANG_ORDER))}


def k_colors(ks):
    return {k: K_RAMP[min(i, len(K_RAMP) - 1)] for i, k in enumerate(sorted(ks))}


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

## Load everything

One tidy table per source. `members` holds candidate-ranking EM (macro over relations) for every
member (zero-shot and each `P<k>_s<seed>` prompt), every length-normalisation α, and the
all / single-token / multi-token subsets; `fit` is the validation split used to pick α and fit the
ensemble weights, `test` is held out.

In [ ]:
EVAL_FILE = re.compile(r"^eval_P(\d+)_seed(\d+)\.json$")
MEMBER = re.compile(r"^P(\d+)_s(\d+)$")
HISTORY_DIR = re.compile(r"^P(\d+)_seed(\d+)$")


def lang_dirs(results_root):
    if not results_root.is_dir():
        return
    for model_dir in sorted(p for p in results_root.iterdir() if p.is_dir()):
        for lang_dir in sorted(p for p in model_dir.iterdir() if p.is_dir()):
            yield model_dir.name, lang_dir.name, lang_dir


eval_rows, member_rows, system_rows, weight_rows, selection_rows, history_rows = [], [], [], [], [], []

for variant, paths in VARIANTS.items():
    for model, lang, folder in lang_dirs(paths["results"]):
        # Per-prompt evaluation: teacher-forced token accuracy at the gold masks.
        for file in sorted(folder.glob("eval_P*_seed*.json")):
            k, seed = map(int, EVAL_FILE.match(file.name).groups())
            result = json.loads(file.read_text(encoding="utf-8"))
            zero = result.get("zero_shot", {}).get("accuracy", np.nan)
            eval_rows.append(dict(variant=variant, model=model, lang=lang, k=k, seed=seed,
                                  token_acc_zero_shot=zero,
                                  token_acc_prompt=result["prompt_tuned"]["accuracy"],
                                  n_test=result["metadata"]["num_evaluation_examples"]))

        # Candidate ranking, length normalisation and ensembles.
        ens_file = folder / "ensemble.json"
        if not ens_file.exists():
            continue
        ens = json.loads(ens_file.read_text(encoding="utf-8"))
        selected = float(ens["summary"]["selected_alpha"])
        selection_rows.append(dict(variant=variant, model=model, lang=lang, alpha=selected,
                                   best_member=ens["summary"]["best_member"],
                                   n_test=ens["metadata"]["splits"]["eval"]["num_scored"]))

        for alpha_key, entry in ens["by_alpha"].items():
            alpha = float(alpha_key)
            for member, res in entry["members"].items():
                match = MEMBER.match(member)
                k, seed = (int(match.group(1)), int(match.group(2))) if match else (0, np.nan)
                for subset in ("all", "single", "multi"):
                    member_rows.append(dict(variant=variant, model=model, lang=lang, alpha=alpha,
                                            member=member, k=k, seed=seed, subset=subset,
                                            fit=res["fit"][subset]["macro"],
                                            test=res["eval"][subset]["macro"],
                                            test_micro=res["eval"][subset]["micro"],
                                            n=res["eval"][subset]["n"]))
            for system in ("uniform", "learned"):
                if system in entry:
                    for subset in ("all", "single", "multi"):
                        system_rows.append(dict(variant=variant, model=model, lang=lang, alpha=alpha,
                                                system=system, subset=subset,
                                                fit=entry[system]["fit"][subset]["macro"],
                                                test=entry[system]["eval"][subset]["macro"]))
            if alpha == selected and "learned" in entry:
                for member, weight in entry["learned"]["weights"].items():
                    weight_rows.append(dict(variant=variant, model=model, lang=lang,
                                            member=member, weight=weight))

    # Training curves.
    ckpt_root = paths["checkpoints"]
    if ckpt_root.is_dir():
        for history in ckpt_root.glob("*/*/P*_seed*/history_*.json"):
            k, seed = map(int, HISTORY_DIR.match(history.parent.name).groups())
            model, lang = history.parent.parent.parent.name, history.parent.parent.name
            for row in json.loads(history.read_text(encoding="utf-8")):
                history_rows.append(dict(variant=variant, model=model, lang=lang, k=k, seed=seed, **row))

evals = pd.DataFrame(eval_rows)
members = pd.DataFrame(member_rows)
systems = pd.DataFrame(system_rows)
weights = pd.DataFrame(weight_rows)
selection = pd.DataFrame(selection_rows)
history = pd.DataFrame(history_rows)

for name, frame in [("evals", evals), ("members", members), ("systems", systems),
                    ("weights", weights), ("selection", selection), ("history", history)]:
    print(f"{name:<10} {len(frame):>6} rows")

assert len(members), "No ensemble.json files found - check the paths printed above."
VARIANT_NAMES = [v for v in VARIANTS if v in set(members.variant)]
MODELS = ordered(members.model, MODEL_ORDER)
LANGS = ordered(members.lang, LANG_ORDER)
KS = sorted(int(k) for k in members.k.unique() if k > 0)
LANG_COLORS = lang_colors(LANGS)
K_COLORS = k_colors(KS)
SEEDS_BY_VARIANT = (members[members.k > 0].groupby("variant").seed.unique()
                    .apply(lambda s: sorted(int(x) for x in s)).to_dict())
print("variants:", VARIANT_NAMES, "| models:", MODELS, "| langs:", LANGS, "| k:", KS)
print("seeds per variant:", SEEDS_BY_VARIANT)
# Figures 1, 2, 4 and 4b plot the two main variants (both models); the follow-up
# experiments (one model each) appear in the tables only.
PLOT_VARIANTS = [v for v in ("keep", "drop") if v in VARIANT_NAMES] or VARIANT_NAMES

# Member EM at the alpha that validation selected for that (variant, model, lang).
at_selected = members.merge(selection[["variant", "model", "lang", "alpha"]],
                            on=["variant", "model", "lang", "alpha"])

## Table 0 — Discrete-template baseline (for reference)

Exact-match accuracy from `scripts/probe.py` + `scripts/ana.py` with the manual templates.
**Not directly comparable** with the numbers below: the baseline decodes freely and matches
aliases, while the ablation ranks each relation's candidate objects.

In [ ]:
if BASELINE.exists():
    baseline = pd.read_csv(BASELINE, sep="\t")
    display(baseline.round(3))
else:
    print(f"No baseline at {BASELINE}")

## Figure 1 — Capacity: does a longer prompt help?

Test EM of each trained prompt (candidate ranking, at the validation-selected α), per language,
with the language average as a dashed line. With several seeds, points are the mean over seeds
and error bars show ±1 standard deviation; with one seed, differences of 1–3 points within a
language are within noise.

In [ ]:
per_seed = at_selected[(at_selected.subset == "all") & (at_selected.k > 0)]
capacity = (per_seed.groupby(["variant", "model", "lang", "k"], as_index=False)
            .agg(test=("test", "mean"), std=("test", "std"), n_seeds=("seed", "nunique")))

fig, axes = plt.subplots(len(PLOT_VARIANTS), len(MODELS), figsize=(5 * len(MODELS), 3.6 * len(PLOT_VARIANTS)),
                         sharey=True, squeeze=False)
positions = {k: i for i, k in enumerate(KS)}
for r, variant in enumerate(PLOT_VARIANTS):
    for c, model in enumerate(MODELS):
        ax = axes[r, c]
        data = capacity[(capacity.variant == variant) & (capacity.model == model)]
        if data.empty:  # e.g. a follow-up experiment that trained only the other model
            ax.text(0.5, 0.5, "no runs", transform=ax.transAxes, ha="center", va="center", color=MUTED)
            ax.set_title(f"{MODEL_LABELS.get(model, model)} — {variant}")
            ax.set_xticks([])
            continue
        for lang in LANGS:
            d = data[data.lang == lang].sort_values("k")
            ax.errorbar(d.k.map(positions), d.test, yerr=d["std"].fillna(0) if (d.n_seeds > 1).any() else None,
                        marker="o", color=LANG_COLORS[lang], label=lang, elinewidth=1, capsize=0)
        mean = data.groupby("k").test.mean()
        ax.plot([positions[k] for k in mean.index], mean.values, color=INK, linestyle="--",
                linewidth=1.5, label="mean")
        # One selective label: the language mean at the longest prompt.
        ax.annotate(f"mean {mean.iloc[-1]:.3f}", (positions[mean.index[-1]], mean.iloc[-1]),
                    textcoords="offset points", xytext=(8, 0), va="center", color=INK, fontsize=8)
        ax.set_xlim(-0.25, len(KS) - 0.35)
        ax.set_xticks(range(len(KS)), [str(k) for k in KS])
        ax.set_xlabel("virtual tokens k")
        ax.set_title(f"{MODEL_LABELS.get(model, model)} — {variant}")
        if c == 0:
            ax.set_ylabel("test EM (macro)")
axes[0, -1].legend(loc="upper left", bbox_to_anchor=(1.01, 1))
fig.tight_layout()
save(fig, "1_capacity_em")
plt.show()

display(Markdown("**Test EM by k** — mean over seeds (rows: variant / model / language)"))
display(capacity.pivot_table(index=["variant", "model", "lang"], columns="k", values="test").round(3))
if (capacity.n_seeds > 1).any():
    display(Markdown("**Standard deviation over seeds**"))
    display(capacity.pivot_table(index=["variant", "model", "lang"], columns="k", values="std").round(3))

In [ ]:
# Same question with the teacher-forced token accuracy from prompt_tuning_eval.py.
if len(evals):
    token = evals.groupby(["variant", "model", "k"], as_index=False)[["token_acc_zero_shot", "token_acc_prompt"]].mean()
    token["gain"] = token.token_acc_prompt - token.token_acc_zero_shot
    display(Markdown("**Token accuracy averaged over languages**"))
    display(token.pivot_table(index=["variant", "model"], columns="k",
                              values=["token_acc_prompt", "gain"]).round(3))

## Figure 2 — Soft-prompt gains: mBERT (WordPiece) vs XLM-R (SentencePiece)

Gain = prompt EM − zero-shot EM for the same model, language and test facts.
Both models were trained with the same hyper-parameters (learning rate 0.3), so part of a gap
can be a tuning effect rather than a tokenizer effect.

In [ ]:
zero = (at_selected[(at_selected.subset == "all") & (at_selected.member == "zero_shot")]
        [["variant", "model", "lang", "test"]].rename(columns={"test": "zero_shot"}))
seed_gains = per_seed[["variant", "model", "lang", "k", "seed", "test"]].merge(zero, on=["variant", "model", "lang"])
seed_gains["gain"] = seed_gains.test - seed_gains.zero_shot
gains = (seed_gains.groupby(["variant", "model", "lang", "k"], as_index=False)
         .agg(zero_shot=("zero_shot", "first"), test=("test", "mean"), gain=("gain", "mean"),
              gain_std=("gain", "std"), n_seeds=("seed", "nunique")))

fig, axes = plt.subplots(len(PLOT_VARIANTS), len(KS), figsize=(4.2 * len(KS), 3.4 * len(PLOT_VARIANTS)),
                         sharey=True, squeeze=False)
width = 0.38
x = np.arange(len(LANGS) + 1)  # languages + average
for r, variant in enumerate(PLOT_VARIANTS):
    for c, k in enumerate(KS):
        ax = axes[r, c]
        for i, model in enumerate(MODELS):
            d = gains[(gains.variant == variant) & (gains.k == k) & (gains.model == model)].set_index("lang")
            values = [d.gain.get(lang, np.nan) for lang in LANGS] + [d.gain.mean()]
            errors = [d.gain_std.get(lang, np.nan) for lang in LANGS] + [np.nan]
            ax.bar(x + (i - (len(MODELS) - 1) / 2) * width, values, width, color=MODEL_COLORS[model],
                   edgecolor=SURFACE, linewidth=1.5, label=MODEL_LABELS.get(model, model),
                   yerr=errors if np.isfinite(errors).any() else None,
                   error_kw=dict(ecolor=INK2, elinewidth=1, capsize=0))
        ax.axhline(0, color=AXIS, linewidth=1)
        ax.set_xticks(x, LANGS + ["avg"])
        ax.set_title(f"k = {k} — {variant}")
        if c == 0:
            ax.set_ylabel("EM gain over zero-shot")
axes[0, -1].legend(loc="upper left", bbox_to_anchor=(1.01, 1))
fig.tight_layout()
save(fig, "2_model_gains")
plt.show()

display(Markdown("**Average over languages**"))
display(gains.groupby(["variant", "model", "k"])[["zero_shot", "test", "gain"]].mean()
        .rename(columns={"test": "prompt_em"}).round(3))

In [ ]:
# Table 2b - tokenizer view: how many test facts have a single-token vs multi-token gold answer per model.
counts = (at_selected[(at_selected.member == "zero_shot") & (at_selected.subset != "all")]
          .pivot_table(index=["variant", "lang"], columns=["model", "subset"], values="n"))
display(Markdown("**Test facts by gold answer length** (SentencePiece usually splits more names into several pieces)"))
display(counts)

## Figure 3 — Length-weighted score normalisation

Candidate score = Σ log p(token) / L<sup>α</sup>. α = 0 is the raw sum (long candidates decay
exponentially), α = 1 the per-token mean. Lines are averaged over languages; one line per k.

In [ ]:
norm = (members[(members.variant == FOCUS) & (members.k > 0)]
        .groupby(["model", "subset", "k", "alpha"], as_index=False).test.mean())
subsets = ["single", "multi", "all"]
fig, axes = plt.subplots(len(MODELS), 3, figsize=(12, 3.3 * len(MODELS)), sharex=True, squeeze=False)
for r, model in enumerate(MODELS):
    for c, subset in enumerate(subsets):
        ax = axes[r, c]
        for k in KS:
            d = norm[(norm.model == model) & (norm.subset == subset) & (norm.k == k)].sort_values("alpha")
            ax.plot(d.alpha, d.test, marker="o", color=K_COLORS[k], label=f"k = {k}")
        ax.set_title(f"{MODEL_LABELS.get(model, model)} — {subset}-token")
        if r == len(MODELS) - 1:
            ax.set_xlabel("α")
        if c == 0:
            ax.set_ylabel(f"test EM ({FOCUS})")
axes[0, -1].legend(loc="upper left", bbox_to_anchor=(1.01, 1))
fig.tight_layout()
save(fig, f"3_length_normalisation_{FOCUS}")
plt.show()

display(Markdown("**α selected on validation** (count of model × language runs)"))
display(selection.pivot_table(index="variant", columns="alpha", values="lang", aggfunc="count").fillna(0).astype(int))

raw = members[(members.alpha == 0) & (members.subset == "multi")][["variant", "model", "lang", "member", "test"]]
best = (at_selected[at_selected.subset == "multi"]
        .merge(selection[["variant", "model", "lang", "best_member"]],
               left_on=["variant", "model", "lang", "member"],
               right_on=["variant", "model", "lang", "best_member"]))
effect = best.merge(raw, on=["variant", "model", "lang", "member"], suffixes=("_normalised", "_raw"))
effect["change"] = effect.test_normalised - effect.test_raw
display(Markdown("**Best single prompt, multi-token EM: raw sum (α = 0) vs selected α**"))
display(effect[["variant", "model", "lang", "member", "alpha", "test_raw", "test_normalised", "change"]]
        .sort_values(["variant", "model", "lang"]).round(3).reset_index(drop=True))

## Figure 4 — Ensembling with learnable scalar weights

All systems at the validation-selected α. *Best single prompt* is chosen on validation, so it is a
fair, strong baseline for the ensemble.

In [ ]:
rows = []
for _, sel in selection.iterrows():
    key = (members.variant == sel.variant) & (members.model == sel.model) & \
          (members.lang == sel.lang) & (members.alpha == sel.alpha) & (members.subset == "all")
    get = lambda name: members[key & (members.member == name)].test.squeeze()
    sys_key = (systems.variant == sel.variant) & (systems.model == sel.model) & \
              (systems.lang == sel.lang) & (systems.alpha == sel.alpha) & (systems.subset == "all")
    row = dict(variant=sel.variant, model=sel.model, lang=sel.lang,
               zero_shot=get("zero_shot"), best_single=get(sel.best_member))
    for system in ("uniform", "learned"):
        match = systems[sys_key & (systems.system == system)].test
        row[system] = match.squeeze() if len(match) else np.nan
    rows.append(row)
ensembles = pd.DataFrame(rows)
ensembles["learned_minus_best"] = ensembles.learned - ensembles.best_single

fig, axes = plt.subplots(len(PLOT_VARIANTS), len(MODELS), figsize=(5.5 * len(MODELS), 3.4 * len(PLOT_VARIANTS)),
                         sharey=True, squeeze=False)
width = 0.2
x = np.arange(len(LANGS))
for r, variant in enumerate(PLOT_VARIANTS):
    for c, model in enumerate(MODELS):
        ax = axes[r, c]
        d = ensembles[(ensembles.variant == variant) & (ensembles.model == model)].set_index("lang")
        for i, system in enumerate(SYSTEM_ORDER):
            values = [d[system].get(lang, np.nan) for lang in LANGS]
            ax.bar(x + (i - 1.5) * width, values, width, color=SYSTEM_COLORS[system],
                   edgecolor=SURFACE, linewidth=1.5, label=SYSTEM_LABELS[system])
        ax.set_xticks(x, LANGS)
        ax.set_title(f"{MODEL_LABELS.get(model, model)} — {variant}")
        if c == 0:
            ax.set_ylabel("test EM (macro)")
axes[0, -1].legend(loc="upper left", bbox_to_anchor=(1.01, 1))
fig.tight_layout()
save(fig, "4_ensembles")
plt.show()

display(ensembles.round(3))
display(Markdown("**Mean over languages**"))
display(ensembles.groupby(["variant", "model"])[["zero_shot", "best_single", "uniform", "learned",
                                                 "learned_minus_best"]].mean().round(4))

## Figure 4b — Learned ensemble weights

Softmax-normalised scalar weight of each member, fit on validation at the selected α.
A row dominated by one member means the ensemble behaves like selecting that prompt.

In [ ]:
if weights.empty:
    print("No learned weights found: ensemble.json files need at least two members.")
else:
    def member_key(m):
        match = MEMBER.match(m)
        return tuple(map(int, match.groups())) if match else (0, 0)

    member_order = ["zero_shot"] + sorted((m for m in weights.member.unique() if m != "zero_shot"), key=member_key)
    tables = {v: weights[weights.variant == v].pivot_table(index=["model", "lang"], columns="member", values="weight")
                 .reindex(columns=member_order)
              for v in PLOT_VARIANTS if (weights.variant == v).any()}
    rows = max(len(t) for t in tables.values())

    fig, axes = plt.subplots(1, len(tables), figsize=(1.1 * len(member_order) * len(tables) + 2.5, 0.42 * rows + 1.4),
                             squeeze=False)
    for c, (ax, (variant, w)) in enumerate(zip(axes[0], tables.items())):
        image = ax.imshow(w.values, cmap=BLUES, vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(len(member_order)), member_order, rotation=30, ha="right")
        # Same rows in every panel: label them once, on the left.
        labels = [f"{MODEL_LABELS.get(m, m).split()[0]} {l}" for m, l in w.index] if c == 0 else []
        ax.set_yticks(range(len(w)), labels)
        ax.grid(False)
        for i in range(w.shape[0]):
            for j in range(w.shape[1]):
                value = w.values[i, j]
                if not np.isnan(value):
                    ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8,
                            color="#ffffff" if value > 0.55 else INK)
        ax.set_title(f"Learned weights — {variant}")
    fig.subplots_adjust(wspace=0.06)
    fig.colorbar(image, ax=axes[0].tolist(), fraction=0.03, label="weight")
    save(fig, "4b_weights")
    plt.show()

    display(Markdown("**Largest weight per run**"))
    top = weights.loc[weights.groupby(["variant", "model", "lang"]).weight.idxmax()]
    display(top.rename(columns={"member": "top_member", "weight": "top_weight"})
            .reset_index(drop=True).round(2))

## Figure 5 — Keep vs drop

**Caveat:** the two variants are evaluated on different test sets (`drop` removes facts), so a
difference mixes *training on cleaner data* with *testing on easier/cleaner facts*. Languages
without inflection (en, nl, ko) show how much the test-set change alone moves the numbers.
A fair comparison evaluates the `keep` prompts on the `drop` test set (it is a subset).

In [ ]:
if {"keep", "drop"} <= set(VARIANT_NAMES):
    both = (at_selected[(at_selected.subset == "all") & at_selected.variant.isin(["keep", "drop"])]
            .pivot_table(index=["model", "lang", "member", "k"], columns="variant", values="test")
            .dropna().reset_index())
    both["drop_minus_keep"] = both["drop"] - both["keep"]

    fig, axes = plt.subplots(1, len(MODELS), figsize=(5.5 * len(MODELS), 3.4), sharey=True, squeeze=False)
    for c, model in enumerate(MODELS):
        ax = axes[0, c]
        d = both[both.model == model]
        for i, lang in enumerate(LANGS):
            values = d[d.lang == lang].drop_minus_keep
            ax.scatter(np.full(len(values), i) + np.linspace(-0.15, 0.15, len(values)), values,
                       s=40, color=LANG_COLORS[lang], edgecolor=SURFACE, linewidth=1.5, zorder=3)
            ax.plot([i - 0.25, i + 0.25], [values.mean()] * 2, color=INK, linewidth=2)
        ax.axhline(0, color=AXIS, linewidth=1)
        ax.set_xticks(range(len(LANGS)), LANGS)
        ax.set_title(MODEL_LABELS.get(model, model))
        if c == 0:
            ax.set_ylabel("EM: drop − keep")
    fig.suptitle("Dots: zero-shot and each k; bar: mean", color=INK2, fontsize=9, y=1.0)
    fig.tight_layout()
    save(fig, "5_keep_vs_drop")
    plt.show()

    display(both.pivot_table(index=["model", "lang"], columns="member",
                             values="drop_minus_keep").round(3))
else:
    print("Need both variants for this section.")

## Figure 6 — Training curves

Validation loss per epoch (the best-validation-loss checkpoint is the one evaluated).
A best epoch at the last epoch suggests the prompt is still improving.

In [ ]:
if len(history):
    h = history[history.variant == FOCUS]
    fig, axes = plt.subplots(len(MODELS), len(LANGS), figsize=(2.9 * len(LANGS), 2.6 * len(MODELS)),
                             sharex=True, squeeze=False)
    for r, model in enumerate(MODELS):
        for c, lang in enumerate(LANGS):
            ax = axes[r, c]
            for k in KS:
                d = h[(h.model == model) & (h.lang == lang) & (h.k == k)].groupby("epoch").val_loss.mean()
                ax.plot(d.index, d.values, marker="o", markersize=4, color=K_COLORS[k], label=f"k = {k}")
            ax.set_title(f"{MODEL_LABELS.get(model, model).split()[0]} {lang}", fontsize=9)
            if c == 0:
                ax.set_ylabel("val loss")
            if r == len(MODELS) - 1:
                ax.set_xlabel("epoch")
    axes[0, -1].legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    fig.tight_layout()
    save(fig, f"6_training_curves_{FOCUS}")
    plt.show()

    best_epoch = (history.loc[history.groupby(["variant", "model", "lang", "k", "seed"]).val_loss.idxmin()]
                  [["variant", "model", "lang", "k", "seed", "epoch", "val_loss", "val_acc"]])
    best_epoch["last_epoch"] = best_epoch.merge(
        history.groupby(["variant", "model", "lang", "k", "seed"], as_index=False).epoch.max(),
        on=["variant", "model", "lang", "k", "seed"], suffixes=("", "_max")).epoch_max.values
    display(Markdown("**Best epoch per run**"))
    display(best_epoch.pivot_table(index=["variant", "model", "lang"], columns="k", values="epoch"))
    print(f"Runs whose best epoch is the last epoch: "
          f"{(best_epoch.epoch == best_epoch.last_epoch).sum()} / {len(best_epoch)}")
else:
    print("No training histories found (copy checkpoints/*/history_*.json to see them).")

## Figure 6b — XLM-R: learning rate 0.3 vs 0.05

Best (lowest validation loss) epoch per run and validation-loss curves for the two learning
rates; the Dutch table checks whether short prompts learned at all.

In [ ]:
if not (ROOT / "checkpoints" / "ablation_keep_xlmr_lr005").is_dir():
    print("No lr 0.05 run found (checkpoints/ablation_keep_xlmr_lr005) - skipping.")
else:
    # XLM-R training stability: learning rate 0.3 (original keep run) vs 0.05.
    # Reads checkpoints/<run>/xlmr_base/<lang>/P<k>_seed<s>/history_<lang>_P<k>.json
    LR_RUNS = {
        "lr 0.3": ROOT / "checkpoints" / "ablation_keep",
        "lr 0.05": ROOT / "checkpoints" / "ablation_keep_xlmr_lr005",
    }

    rows = []
    for run, root in LR_RUNS.items():
        for file in root.glob("xlmr_base/*/P*_seed*/history_*.json"):
            k, seed = map(int, HISTORY_DIR.match(file.parent.name).groups())
            for row in json.loads(file.read_text(encoding="utf-8")):
                rows.append(dict(run=run, lang=file.parent.parent.name, k=k, seed=seed, **row))
    lr_hist = pd.DataFrame(rows)
    assert len(lr_hist), "No XLM-R histories found - check the folders in LR_RUNS."

    runs = [r for r in LR_RUNS if r in set(lr_hist.run)]
    langs = ordered(lr_hist.lang, LANG_ORDER)
    ks = sorted(lr_hist.k.unique())
    colors = k_colors(ks)
    last_epoch = lr_hist.groupby("run").epoch.max()

    # Best (lowest validation loss) epoch per run - this checkpoint is the one evaluated.
    best = lr_hist.loc[lr_hist.groupby(["run", "lang", "k", "seed"]).val_loss.idxmin()]
    display(Markdown("**Best epoch per run** (lowest validation loss)"))
    display(best.pivot_table(index="lang", columns=["run", "k"], values="epoch").reindex(langs))

    summary = best.groupby("run").agg(runs=("epoch", "size"),
                                      best_at_epoch_1_or_2=("epoch", lambda e: int((e <= 2).sum())))
    summary["best_at_last_epoch"] = [int((best[best.run == r].epoch == last_epoch[r]).sum()) for r in summary.index]
    display(summary)

    # Did the weak Dutch prompts learn at all? Validation accuracy per epoch.
    display(Markdown("**Dutch, k = 5 and 10: validation accuracy per epoch**"))
    display(lr_hist[(lr_hist.lang == "nl") & (lr_hist.k.isin([5, 10]))]
            .pivot_table(index="epoch", columns=["run", "k"], values="val_acc").round(3))

    # Validation-loss curves: rows = learning rate, columns = language, one line per k.
    fig, axes = plt.subplots(len(runs), len(langs), figsize=(2.9 * len(langs), 2.6 * len(runs)),
                             sharex=True, squeeze=False)
    for r, run in enumerate(runs):
        for c, lang in enumerate(langs):
            ax = axes[r, c]
            for k in ks:
                d = lr_hist[(lr_hist.run == run) & (lr_hist.lang == lang) & (lr_hist.k == k)].groupby("epoch").val_loss.mean()
                ax.plot(d.index, d.values, marker="o", markersize=4, color=colors[k], label=f"k = {k}")
            ax.set_title(f"XLM-R {lang} — {run}", fontsize=9)
            if c == 0:
                ax.set_ylabel("val loss")
            if r == len(runs) - 1:
                ax.set_xlabel("epoch")
    axes[0, -1].legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    fig.tight_layout()
    save(fig, "6b_xlmr_learning_rate")
    plt.show()

## Figure 7 — Soft vs discrete prompts, per language

**Discrete** = X-FACTR's manual template, no training, scored two ways: free decoding with alias
matching (Table 0, `probe.py` + `ana.py`) and candidate ranking (the zero-shot model).
**Soft** = trained soft prompt, scored by candidate ranking; for each model and language the
prompt length k with the best mean *validation* EM is shown, as mean ± std over seeds.
Soft vs discrete-ranking is the like-for-like comparison; free decoding is a harder protocol.

In [ ]:
DISCRETE_SOFT_COLORS = {"Discrete — free decoding": CATEGORICAL[1],
                        "Discrete — candidate ranking": CATEGORICAL[2],
                        "Soft prompt — candidate ranking": CATEGORICAL[0]}

# Soft prompt: per (variant, model, lang) pick k by mean validation EM, report test mean +- std.
fit_mean = (per_seed.groupby(["variant", "model", "lang", "k"], as_index=False)
            .agg(fit=("fit", "mean"), test=("test", "mean"), test_std=("test", "std"),
                 n_seeds=("seed", "nunique")))
soft = fit_mean.loc[fit_mean.groupby(["variant", "model", "lang"]).fit.idxmax()]
compare = soft.merge(zero, on=["variant", "model", "lang"]).rename(
    columns={"k": "best_k", "test": "soft_EM", "test_std": "soft_EM_std", "zero_shot": "discrete_ranking_EM"})
if BASELINE.exists():
    compare = compare.merge(baseline[["model", "lang", "em_all"]].rename(columns={"em_all": "discrete_decoding_EM"}),
                            on=["model", "lang"], how="left")
else:
    compare["discrete_decoding_EM"] = np.nan
compare["soft_minus_discrete_ranking"] = compare.soft_EM - compare.discrete_ranking_EM

data = compare[compare.variant == FOCUS]
models = [m for m in MODELS if m in set(data.model)]
fig, axes = plt.subplots(1, len(models), figsize=(5.5 * len(models), 3.6), sharey=True, squeeze=False)
width = 0.26
x = np.arange(len(LANGS))
columns = {"Discrete — free decoding": ("discrete_decoding_EM", None),
           "Discrete — candidate ranking": ("discrete_ranking_EM", None),
           "Soft prompt — candidate ranking": ("soft_EM", "soft_EM_std")}
for c, model in enumerate(models):
    ax = axes[0, c]
    d = data[data.model == model].set_index("lang")
    for i, (label, (value_col, err_col)) in enumerate(columns.items()):
        values = [d[value_col].get(lang, np.nan) for lang in LANGS]
        errors = [d[err_col].get(lang, np.nan) for lang in LANGS] if err_col else None
        has_err = errors is not None and np.isfinite(errors).any()
        ax.bar(x + (i - 1) * width, values, width, color=DISCRETE_SOFT_COLORS[label],
               edgecolor=SURFACE, linewidth=1.5, label=label,
               yerr=errors if has_err else None, error_kw=dict(ecolor=INK2, elinewidth=1, capsize=0))
    ax.set_xticks(x, LANGS)
    ax.set_title(f"{MODEL_LABELS.get(model, model)} — {FOCUS}")
    if c == 0:
        ax.set_ylabel("test EM (macro)")
axes[0, -1].legend(loc="upper left", bbox_to_anchor=(1.01, 1))
fig.tight_layout()
save(fig, f"7_soft_vs_discrete_{FOCUS}")
plt.show()

display(compare[["variant", "model", "lang", "discrete_decoding_EM", "discrete_ranking_EM", "best_k",
                 "soft_EM", "soft_EM_std", "n_seeds", "soft_minus_discrete_ranking"]]
        .sort_values(["variant", "model", "lang"]).reset_index(drop=True).round(3))

In [ ]:
# Manual prompting (X-FACTR templates, no training) - both evaluations side by side.
manual = (ensembles[ensembles.variant == "keep"][["model", "lang", "zero_shot"]]
          .rename(columns={"zero_shot": "candidate_EM"}))
if len(evals):
    token = (evals[evals.variant == "keep"].groupby(["model", "lang"], as_index=False)
             .token_acc_zero_shot.first().rename(columns={"token_acc_zero_shot": "token_acc"}))
    manual = manual.merge(token, on=["model", "lang"], how="left")
if BASELINE.exists():
    manual = manual.merge(baseline[["model", "lang", "em_all", "em_single", "em_multi"]]
                          .rename(columns={"em_all": "decoding_EM", "em_single": "decoding_EM_single",
                                           "em_multi": "decoding_EM_multi"}),
                          on=["model", "lang"], how="left")
manual["lang"] = pd.Categorical(manual.lang, ordered(manual.lang, LANG_ORDER), ordered=True)
display(Markdown("**Manual prompting** — free decoding with alias matching (`decoding_*`, Table 0) "
                 "vs candidate ranking (`candidate_EM`) vs teacher-forced token accuracy (`token_acc`); "
                 "keep test split"))
display(manual.sort_values(["model", "lang"]).reset_index(drop=True).round(3))
manual.to_csv(FIG_DIR / "manual_prompting.csv", index=False)

## Figure 8 — Learning-rate × schedule sweep (k = 20)

From `bash run_lr_sweep.sh`: both models at k = 20, all languages, 3 seeds, for each peak learning
rate and schedule (`linear` = warmup + linear decay, the original setup; `plateau` = warmup, then halve
the rate and restore the best prompt when validation loss stalls). Each model's configuration is
chosen on **validation** EM; only then is its test EM reported. Stability columns count runs whose
best epoch is 1–2 and runs whose final validation accuracy fell below half of their peak (collapse).

In [ ]:
SWEEP_RESULTS = ROOT / "results" / "lr_sweep"
SWEEP_CKPTS = ROOT / "checkpoints" / "lr_sweep"
SWEEP_DIR = re.compile(r"^(linear|plateau)_lr([0-9.]+)$")
SCHEDULE_COLORS = {"linear": CATEGORICAL[0], "plateau": CATEGORICAL[1]}

rows, curve_rows = [], []
configs = [(d, SWEEP_DIR.match(d.name)) for d in sorted(SWEEP_RESULTS.iterdir())] if SWEEP_RESULTS.is_dir() else []
for cfg_dir, match in configs:
    if not match:
        continue
    schedule, lr = match.group(1), float(match.group(2))
    for model, lang, folder in lang_dirs(cfg_dir):
        ens_file = folder / "ensemble.json"
        if not ens_file.exists():
            continue
        ens = json.loads(ens_file.read_text(encoding="utf-8"))
        selected = float(ens["summary"]["selected_alpha"])
        entry = next(e for key, e in ens["by_alpha"].items() if float(key) == selected)
        for member, res in entry["members"].items():
            m = MEMBER.match(member)
            if not m:
                continue
            k, seed = map(int, m.groups())
            rows.append(dict(schedule=schedule, lr=lr, model=model, lang=lang, k=k, seed=seed,
                             val=res["fit"]["all"]["macro"], test=res["eval"]["all"]["macro"],
                             test_multi=res["eval"]["multi"]["macro"]))
    # Training curves, for stability counts.
    for history_file in (SWEEP_CKPTS / cfg_dir.name).glob("*/*/P*_seed*/history_*.json"):
        k, seed = map(int, HISTORY_DIR.match(history_file.parent.name).groups())
        h = pd.DataFrame(json.loads(history_file.read_text(encoding="utf-8")))
        curve_rows.append(dict(schedule=schedule, lr=lr, model=history_file.parent.parent.parent.name,
                               lang=history_file.parent.parent.name, k=k, seed=seed,
                               best_epoch=int(h.loc[h.val_loss.idxmin(), "epoch"]),
                               last_epoch=int(h.epoch.max()),
                               collapsed=bool(h.val_acc.iloc[-1] < 0.5 * h.val_acc.max())))

if not rows:
    print(f"No sweep results under {SWEEP_RESULTS} (run bash run_lr_sweep.sh) - skipping.")
else:
    sweep = pd.DataFrame(rows)
    # Mean over languages per seed, then mean +- std over seeds.
    per_seed_cfg = sweep.groupby(["model", "schedule", "lr", "seed"], as_index=False)[["val", "test", "test_multi"]].mean()
    table = (per_seed_cfg.groupby(["model", "schedule", "lr"], as_index=False)
             .agg(val=("val", "mean"), val_std=("val", "std"), test=("test", "mean"),
                  test_std=("test", "std"), test_multi=("test_multi", "mean"), n_seeds=("seed", "nunique")))
    if curve_rows:
        curves = pd.DataFrame(curve_rows)
        stability = (curves.assign(early=curves.best_epoch <= 2)
                     .groupby(["model", "schedule", "lr"], as_index=False)
                     .agg(runs=("seed", "size"), best_at_epoch_1_or_2=("early", "sum"),
                          collapsed=("collapsed", "sum")))
        table = table.merge(stability, on=["model", "schedule", "lr"], how="left")

    # Selection on validation only.
    chosen = table.loc[table.groupby("model").val.idxmax()]

    models = ordered(table.model, MODEL_ORDER)
    fig, axes = plt.subplots(1, len(models), figsize=(5.2 * len(models), 3.5), sharey=True, squeeze=False)
    for c, model in enumerate(models):
        ax = axes[0, c]
        for schedule in ("linear", "plateau"):
            d = table[(table.model == model) & (table.schedule == schedule)].sort_values("lr")
            if d.empty:
                continue
            ax.errorbar(d.lr, d.val, yerr=d.val_std if d.n_seeds.max() > 1 else None, marker="o",
                        color=SCHEDULE_COLORS[schedule], label=schedule, elinewidth=1, capsize=0)
        best = chosen[chosen.model == model].iloc[0]
        ax.scatter([best.lr], [best.val], s=160, facecolors="none", edgecolors=INK, linewidth=1.5, zorder=4)
        ax.annotate(f"selected: {best.schedule}, lr {best.lr:g}", (best.lr, best.val),
                    textcoords="offset points", xytext=(0, -14), ha="center", va="top", color=INK, fontsize=8)
        ax.margins(x=0.12, y=0.18)
        ax.set_xscale("log")
        rates = sorted(table.lr.unique())
        ax.set_xticks(rates, [f"{r:g}" for r in rates])
        ax.minorticks_off()
        ax.set_xlabel("peak learning rate")
        ax.set_title(f"{MODEL_LABELS.get(model, model)} — k = 20")
        if c == 0:
            ax.set_ylabel("validation EM (mean over languages)")
    axes[0, -1].legend(title="schedule", loc="upper left", bbox_to_anchor=(1.01, 1))
    fig.tight_layout()
    save(fig, "8_lr_schedule_sweep")
    plt.show()

    display(Markdown("**Sweep summary** — validation and test EM averaged over languages, ± std over seeds; "
                     "stability counts from the training curves"))
    display(table.sort_values(["model", "schedule", "lr"]).reset_index(drop=True).round(4))
    display(Markdown("**Selected on validation** (report the test EM of this configuration)"))
    display(chosen[["model", "schedule", "lr", "val", "test", "test_std", "test_multi"]].reset_index(drop=True).round(4))
    display(Markdown("**Selected configuration, per language** (test EM, mean over seeds)"))
    per_lang = sweep.merge(chosen[["model", "schedule", "lr"]], on=["model", "schedule", "lr"])
    display(per_lang.pivot_table(index="model", columns="lang", values="test")
            .reindex(columns=ordered(per_lang.lang, LANG_ORDER)).round(3))
    table.to_csv(FIG_DIR / "lr_schedule_sweep.csv", index=False)

## Export

All tables used above, as CSV next to the figures.

In [ ]:
for name, frame in [("capacity", capacity), ("gains", gains), ("ensembles", ensembles),
                    ("soft_vs_discrete", compare), ("members_all_alphas", members),
                    ("selection", selection)]:
    frame.to_csv(FIG_DIR / f"{name}.csv", index=False)
print(f"Saved figures and tables to {FIG_DIR}")